In [1]:
import pickle

In [2]:
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer

In [3]:
output_dir_name = "gemini-2.5-flash"

In [4]:
with open(f"{output_dir_name}/titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags_dict = pickle.load(f)

In [5]:
len(titles_with_tags_dict)

1147

In [6]:
with open(f"{output_dir_name}/tags_set.pkl", "rb") as f:
    tags_set = pickle.load(f)

In [7]:
len(tags_set)

1524

In [8]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
tags_list = sorted(tags_set)

In [10]:
tags_embeddings = model.encode(tags_list, normalize_embeddings=True)

In [11]:
output = []

for eps in range(1, 100):
    dbscan = DBSCAN(eps=eps / 100, min_samples=2)
    dbscan.fit(tags_embeddings)

    labels = sorted(set(dbscan.labels_))

    dbscan_clusters = {}
    for cluster_id, tag_name in zip(dbscan.labels_, tags_list):
        data = dbscan_clusters.get(cluster_id)
        if data:
            data.append(tag_name)
        else:
            data = [tag_name]
        dbscan_clusters[cluster_id] = data

    if len(labels) > 100:
        sc = silhouette_score(tags_embeddings, dbscan.fit_predict(tags_embeddings))
        output.append((eps, sc))

df = pd.DataFrame(data=output, columns=["eps", "sc"])

In [12]:
df.sort_values(by=["sc"], ascending=[False]).reset_index(drop=True).head()

,eps,sc
0,64,-0.018628
1,65,-0.019160
2,63,-0.024647
3,62,-0.026584
4,61,-0.027271


In [13]:
eps = df.sort_values(by=["sc"], ascending=[False]).reset_index(drop=True).iloc[0]["eps"]

In [14]:
dbscan = DBSCAN(eps=eps / 100, min_samples=2)

In [15]:
dbscan.fit(tags_embeddings)

,"eps eps: float, default=0.5The maximum distance between two samples for one to be consideredas in the neighborhood of the other. This is not a maximum boundon the distances of points within a cluster. This is the mostimportant DBSCAN parameter to choose appropriately for your data setand distance function. Smaller values generally lead to more clusters.",np.float64(0.64)
,"min_samples min_samples: int, default=5The number of samples (or total weight) in a neighborhood for a point tobe considered as a core point. This includes the point itself. If`min_samples` is set to a higher value, DBSCAN will find denser clusters,whereas if it is set to a lower value, the found clusters will be moresparse.",2
,"metric metric: str, or callable, default='euclidean'The metric to use when calculating distance between instances in afeature array. If metric is a string or callable, it must be one ofthe options allowed by :func:`sklearn.metrics.pairwise_distances` forits metric parameter.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors for DBSCAN... versionadded:: 0.17 metric *precomputed* to accept precomputed sparse matrix.",'euclidean'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function... versionadded:: 0.19",None
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'The algorithm to be used by the NearestNeighbors moduleto compute pointwise distances and find nearest neighbors.'auto' will attempt to decide the most appropriate algorithmbased on the values passed to :meth:`fit` method.See :class:`~sklearn.neighbors.NearestNeighbors` documentation fordetails.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or cKDTree. This can affect the speedof the construction and query, as well as the memory requiredto store the tree. The optimal value dependson the nature of the problem.",30
,"p p: float, default=NoneThe power of the Minkowski metric to be used to calculate distancebetween points. If None, then ``p=2`` (equivalent to the Euclideandistance). When p=1, this is equivalent to Manhattan distance.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [16]:
labels = sorted(set(dbscan.labels_))

In [17]:
len(labels)

168

In [18]:
dbscan_clusters = {}
for cluster_id, tag_name in zip(dbscan.labels_, tags_list):
    data = dbscan_clusters.get(cluster_id)
    if data:
        data.append(tag_name)
    else:
        data = [tag_name]
    dbscan_clusters[cluster_id] = data

In [19]:
len(dbscan_clusters.keys())

168

In [20]:
len(dbscan_clusters[labels[0]])

736

In [21]:
dbscan_clusters[labels[-1]]

['volunteer_management', 'volunteer_work', 'volunteering', 'volunteerism']

In [22]:
with open(f"{output_dir_name}/dbscan_clusters.pkl", "wb") as f:
    pickle.dump(dbscan_clusters, f)